# **2일차 팀 프로젝트: 문서 기반 RAG 시스템 구축**

## 프로젝트 목표
1. 팀에서 선정한 PDF 문서를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 적용
3. 검색 테스트 및 RAG 시스템 구현

## 구현 단계
- 환경 설정 확인
- PDF 문서 로딩
- Child Chunk 생성 및 Qdrant Cloud 저장
- Parent Document 저장
- 검색 테스트
- RAG 시스템 구현 및 테스트

## 0. 환경 변수 설정

In [12]:
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
    print(f"  URL: {os.environ.get('QDRANT_URL')}")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

✓ OpenAI API Key가 설정되었습니다.
✓ Qdrant Cloud 설정이 완료되었습니다.
  URL: https://0637587f-60ae-43db-b9be-985907b86b22.us-east-1-1.aws.cloud.qdrant.io:6333


## 1. PDF 문서 로딩

**TODO: 팀에서 선정한 PDF 파일 경로를 입력하세요**

In [13]:
from langchain_core.documents import Document
import fitz

# TODO: PDF 파일 경로를 입력하세요
# 예시: "../datasets/your_document.pdf"
file_path = "../datasets/2026-2학기_학사안내_자료.pdf"

doc = fitz.open(file_path)
docs = []

# 페이지 단위로 Document 생성 (Parent Document)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text", sort=True)

    # 빈 페이지는 스킵
    if len(text.strip()) < 10:
        continue

    docs.append(
        Document(
            page_content=text,
            metadata={
                "source": file_path.split("/")[-1],
                "page": page_num + 1,
                "parent_id": f"page_{page_num + 1}"
            }
        )
    )

doc.close()

print(f"총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"\n첫 번째 페이지 길이: {len(docs[0].page_content)}자")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

# 첫 페이지 내용 미리보기
print(f"\n첫 페이지 내용 미리보기:")
print(docs[0].page_content[:300] + "...")

총 45개의 페이지(Parent Document) 로드 완료

첫 번째 페이지 길이: 10자
평균 페이지 길이: 1935자

첫 페이지 내용 미리보기:
2026학년도2학기...


## 2. Child Chunk 생성

**TODO: 청킹 전략을 조정해보세요 (선택사항)**
- chunk_size: 각 청크의 크기 (기본 400자)
- chunk_overlap: 청크 간 겹치는 부분 (기본 50자)

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# TODO: 필요시 chunk_size와 chunk_overlap 값을 조정하세요
child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,      # 작은 크기로 정확한 검색
    chunk_overlap=50     # 문맥 유지
)

# Parent를 Child chunk로 분할
child_docs = []

for parent_doc in docs:
    chunks = child_splitter.split_text(parent_doc.page_content)

    for chunk in chunks:
        child_docs.append(
            Document(
                page_content=chunk,
                metadata={
                    "parent_id": parent_doc.metadata["parent_id"],
                    "page": parent_doc.metadata["page"],
                    "source": parent_doc.metadata["source"]
                }
            )
        )

print(f"\n생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

# Child chunk 샘플 확인
print(f"\nChild chunk 샘플 (첫 3개):")
for i in range(min(3, len(child_docs))):
    print(f"\nChunk {i + 1}:")
    print(f"  Parent ID: {child_docs[i].metadata['parent_id']}")
    print(f"  Page: {child_docs[i].metadata['page']}")
    print(f"  Length: {len(child_docs[i].page_content)}자")
    print(f"  Content: {child_docs[i].page_content[:100]}...")


생성된 통계:
  - Parent 문서 수: 45
  - Child chunk 수: 303
  - 평균 chunk/page: 6.7

Child chunk 샘플 (첫 3개):

Chunk 1:
  Parent ID: page_1
  Page: 1
  Length: 10자
  Content: 2026학년도2학기...

Chunk 2:
  Parent ID: page_2
  Page: 2
  Length: 365자
  Content: 【 목   차 】





1. 2026학년도 제2학기 학사일정                                   1


2. 휴학·복학 전과               ...

Chunk 3:
  Parent ID: page_2
  Page: 2
  Length: 367자
  Content: 7. 교양 교육과정 이수기준                                       17


8. 수강신청                                  ...


## 3. Qdrant Cloud에 Child Chunk 저장

**TODO: 컬렉션 이름을 팀 프로젝트에 맞게 변경하세요**

In [16]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

# Qdrant Cloud 클라이언트 생성
client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")
print(f"  URL: {os.getenv('QDRANT_URL')}")

Qdrant Cloud에 연결되었습니다.
  URL: https://0637587f-60ae-43db-b9be-985907b86b22.us-east-1-1.aws.cloud.qdrant.io:6333


In [17]:
# 임베딩 함수
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# TODO: 팀 프로젝트에 맞는 컬렉션 이름으로 변경하세요
# 예시: "team1_healthcare_docs", "team2_legal_docs" 등
collection_name = "SMU_GUIDE_COLLECTION"

# 컬렉션 존재 여부 확인
collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

if existing_collection:
    print(f"컬렉션 '{collection_name}'이 이미 존재합니다.")
    user_input = input("기존 데이터를 삭제하고 새로 추가하시겠습니까? (y/n): ")

    if user_input.lower() == 'y':
        print(f"컬렉션 '{collection_name}' 삭제 중...")
        client.delete_collection(collection_name=collection_name)
        print("컬렉션이 삭제되었습니다.")

        client.create_collection(
            collection_name=collection_name,
            vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
        )
        print(f"컬렉션 '{collection_name}' 생성 완료")
    else:
        print("기존 컬렉션을 사용합니다.")
else:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    print(f"컬렉션 '{collection_name}' 생성 완료")

# 벡터스토어 생성
vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

# Child chunk 추가
uuids = [str(uuid4()) for _ in range(len(child_docs))]
vectorstore.add_documents(documents=child_docs, ids=uuids)

print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")

컬렉션 'SMU_GUIDE_COLLECTION' 생성 완료

303개의 Child chunk가 Qdrant Cloud에 추가되었습니다.


## 4. Parent Document 저장 (Docstore)

In [19]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_id = parent_doc.metadata["parent_id"]
    parent_docstore[parent_id] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")
print(f"\nDocstore 키 예시: {list(parent_docstore.keys())[:5]}")

Docstore에 45개의 Parent 문서 저장 완료

Docstore 키 예시: ['page_1', 'page_2', 'page_3', 'page_4', 'page_5']


## 5. Parent Document Retriever 구현

In [20]:
from typing import List

class ParentDocumentRetriever:
    """
    Parent Document Retriever 직접 구현

    원리:
    1. vectorstore에서 child chunk 검색
    2. child chunk의 parent_id 추출
    3. docstore에서 parent_id로 parent 문서 반환
    """

    def __init__(self, vectorstore, parent_docstore, k: int = 2):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k

    def invoke(self, query: str) -> List[Document]:
        # 1. Vectorstore에서 child chunk 검색
        child_results = self.vectorstore.similarity_search(query, k=self.k)

        # 2. Child chunk에서 parent_id 추출 (중복 제거)
        parent_ids = []
        for doc in child_results:
            parent_id = doc.metadata.get("parent_id")
            if parent_id and parent_id not in parent_ids:
                parent_ids.append(parent_id)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 parent 문서 가져오기
        parent_docs = []
        for parent_id in parent_ids:
            if parent_id in self.parent_docstore:
                parent_docs.append(self.parent_docstore[parent_id])

        return parent_docs

    def get_child_chunks(self, query: str, k: int = 3) -> List[Document]:
        """비교용: Child chunk 직접 반환"""
        return self.vectorstore.similarity_search(query, k=k)

# Retriever 생성
parent_retriever = ParentDocumentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    k=2
)

print("✓ Parent Document Retriever 생성 완료")

✓ Parent Document Retriever 생성 완료


## 6. 검색 테스트

**TODO: 팀 문서에 맞는 질문으로 변경하여 검색 테스트를 진행하세요**

In [22]:
# TODO: 팀 문서에 맞는 검색 질문을 작성하세요
query = "2학기 교차 수강신청 기간에 대해 알려줘."

print(f"검색 쿼리: {query}\n")
print("="*80)

# Child chunk 검색
print("\n[1] Child Chunk 검색 결과")
print("-"*80)
child_results = parent_retriever.get_child_chunks(query, k=2)

for i, result in enumerate(child_results, start=1):
    print(f"\nChunk {i}:")
    print(f"  페이지: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용: {result.page_content}")

# Parent document 검색
print("\n" + "="*80)
print("\n[2] Parent Document 검색 결과")
print("-"*80)
parent_results = parent_retriever.invoke(query)

for i, result in enumerate(parent_results, start=1):
    print(f"\nPage {i}:")
    print(f"  페이지 번호: {result.metadata.get('page', '?')}")
    print(f"  Parent ID: {result.metadata.get('parent_id')}")
    print(f"  길이: {len(result.page_content)}자")
    print(f"  내용 미리보기: {result.page_content[:300]}...")

검색 쿼리: 2학기 교차 수강신청 기간에 대해 알려줘.


[1] Child Chunk 검색 결과
--------------------------------------------------------------------------------

Chunk 1:
  페이지: 29
  Parent ID: page_29
  길이: 390자
  내용: 4) 기타 안내사항

   가) 교차수강은 양 캠퍼스 정기 수강신청 후 제한인원 여석의 범위 내에서 수강신청 가능

   나) 교차 수강신청 기간에는 상대 캠퍼스 교과목만 수강신청할 수 있으며, 이를 위해 본인 소속 캠퍼스

    신청 학점을 일부 취소는 가능하나, 해당 취소 교과목은 교차수강신청 기간에 재신청할 수 없음

   다) 본인 캠퍼스 개설 교과목과 동일/유사 교과목을 신청한 경우, 학부(과) 및 교무처의 확인을 거쳐

    수강신청이 취소될 수 있음

   라) 교양 교과목 교차수강에 대한 사항은 계당교양교육원 교학지원팀(041-550-5415) 별도 문의

   마) 바이오헬스 교과목 교차수강에 관한 사항은 바이오헬스 혁신융합대학 사업단(02-2287-5146) 별도 문의

Chunk 2:
  페이지: 29
  Parent ID: page_29
  길이: 384자
  내용: 카. 교차수강신청 안내(서울↔천안 캠퍼스간 수강신청)

  1) 교차수강신청 가능학점

   가) 학기당 최대 6학점(계절수업 별도)

   나) 재학기간 최대 21학점(바이오헬스혁신융합교과목, 서울캠퍼스 개설 HUSS교과목은 최대학점 산정 제외)

  2) 대상교과목 및 제외교과목

                                       Ÿ 학생 원 소속 캠퍼스에 개설되지 않는 전공이론(심화, 선택), 일선 교과목 및
          대상 교과목    일반 e-러닝 교과목

                                       Ÿ 콘텐츠제작연계전공 교과목

                       

## 7. RAG 시스템 구현

**TODO: 시스템 프롬프트를 팀 문서에 맞게 수정하세요**

In [23]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# TODO: 시스템 프롬프트를 팀 문서 도메인에 맞게 수정하세요
# 예시: "당신은 의료 전문가입니다.", "당신은 법률 전문가입니다." 등
template = """
당신은  대학교 행정 관계자입니다.
주어진 정보를 바탕으로 사용자의 질문에 정확하고 친절하게 답변하세요.
또한, 답변에 참고한 문서의 출처와 페이지 번호를 명시하세요.
자료에 없는 내용은 지어내지 말고 있는 그대로 대답하세요.

[참고 정보]
{context}

[질문]
{question}

[답변]
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str) -> str:
    """
    Parent Document Retriever를 사용한 RAG
    """
    # 1. 문서 검색
    retrieved_docs = parent_retriever.invoke(question)

    # 2. 컨텍스트 구성
    context_parts = []
    for doc in retrieved_docs:
        source = doc.metadata.get("source", "?")
        page_num = doc.metadata.get("page", "?")
        context_parts.append(f"[출처: {source}, 페이지: {page_num}]\n{doc.page_content}")

    context = "\n\n---\n\n".join(context_parts)

    # 3. 프롬프트 생성
    formatted_prompt = prompt_template.format(
        context=context,
        question=question
    )

    # 4. LLM 호출
    response = llm.invoke(formatted_prompt)
    return response.content

print("✓ RAG 시스템 준비 완료")

✓ RAG 시스템 준비 완료


## 8. RAG 시스템 테스트

**TODO: 팀 문서에 맞는 다양한 질문으로 RAG 시스템을 테스트하세요**

In [24]:
# TODO: 팀 문서에 맞는 질문들을 작성하세요
questions = [
    "들어야하는 교양 수업에 대해 알려줘",
    "장학금에 대한 규정을 찾아줘",
    "수강신청 기간에 대해 알려줘"
]

for q in questions:
    print(f"\n{'='*80}")
    print(f"질문: {q}")
    print(f"{'='*80}\n")

    answer = rag_with_parent_retriever(q)
    display(Markdown(answer))


질문: 들어야하는 교양 수업에 대해 알려줘



교양 수업은 **입학한 당해 연도 기준의 교양교육과정 이수 체계**에 따라 들어야 합니다. 다만, 교육과정이 개편된 경우에는 일부는 소급 적용될 수 있지만, **입학 연도 기준보다 이수해야 할 요건이 늘어나거나 과목 수가 증가한 경우에는 입학 당해 연도 기준을 따릅니다.**  
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

2026학년도 기준으로 교양은 크게 아래처럼 구성됩니다.

### 1) 기초교양
- **사고와표현**: 3학점
- **English for Academic Purposes**: 3학점
- **컴퓨팅사고와데이터의이해**: 2학점
- **문제해결과인공지능**: 2학점
- **교양과인성**: 1학점  
또한 기초교양에는 **교필 4과목 필수 이수**가 표시되어 있고,  
**기초영어/기초수학 중 택1**해야 합니다.  
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

### 2) 상명핵심역량교양
- **5개 역량 중 최소 2개 영역**, 각 1개 과목 이수
- 해당 역량:  
  - 전문지식탐구역량  
  - 창의적문제해결역량  
  - 융복합역량  
  - 다양성존중역량  
  - 윤리실천역량  
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

### 3) 균형교양
- **학과별 미인정영역을 제외한 나머지 영역 중 최소 3개 영역**, 각 1개 과목 이수
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

### 4) 일반교양
- **제한 없음**
- 다만 전체 교양 학점은 **총 33학점 이상**입니다.
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

### 추가로 알아두실 점
- **사고와표현**은 2학년 이내 수강완료를 원칙으로 합니다.
- **기초영어/기초수학 중 택1**도 2학년 이내 수강완료를 원칙으로 합니다.
- **교양과인성**은 1학년만 수강 가능하며, 2학년 이상은 재수강만 가능합니다.
- 출처: **2026-2학기_학사안내_자료.pdf, p.19**

원하시면 제가 다음 답변에서  
**“당장 1학년이 들어야 하는 교양 과목만”** 따로 정리해서 알려드릴게요.


질문: 장학금에 대한 규정을 찾아줘



아래는 **제공해주신 자료에서 확인되는 장학금 관련 규정**입니다.  
다만 자료 범위 내에서 확인 가능한 내용만 정리했으며, 세부 신청 일정·절차 등 **자료에 없는 내용은 포함하지 않았습니다.**

## 1) 교내장학금 규정
교내장학금은 장학금별로 **장학금액, 지급기간, 직전학기 평점평균, 직전학기 취득학점** 등의 지급조건이 다르게 정해져 있습니다.  
예를 들어 자료에는 다음과 같은 장학금이 확인됩니다.

- **재외국민 입학 성적 우수 장학금**
- **상명장학금**
  - 전체수석
  - 단과대학 사랑
  - 학과수석
- **수능우수 장학금**
- **특성화고졸재직자장학금**
- **다전공우수**
- **면학장학금**
- **디딤돌장학금**
- **근로장학금**
- **리더십장학금**
- **체육특기장학금**
- **국가보훈 장학금**
- **북한이탈주민 장학금**
- **형제·자매장학금**

특히 예시로 보면:
- **면학A**: 등록금의 40%, 1학기, 직전학기 평점평균 3.0 이상, 15학점 이상(4학년 9학점 이상)
- **면학B**: 등록금의 40%, 1학기, 직전학기 평점평균 2.0 이상, 12학점 이상(4학년 9학점 이상)
- **근로장학금**: 일정 금액, 1학기, 학사경고 이상, 12학점 이상(4학년 9학점 이상)
- **형제·자매장학금**: 80만원, 재학기간, 직전학기 2.0 이상, 12학점 이상(4학년 9학점 이상)

또한 **“중복수혜 불가”** 문구가 표에 표시되어 있습니다.

## 2) 국가장학금 규정
자료에는 다음 국가장학금 관련 내용이 확인됩니다.

- **인문100년장학금**
  - 등록금 전액
  - 일시 또는 계속지원
  - 대상:
    - 대한민국 국적 소지자 중 대학의 인문사회계열 재학 중인 1학년 신입생
    - 전공탐색유형(4년 지원) 또는 3학년(전공확립유형, 2년 지원) 재학생
- **예술체육비전장학금**
  - 등록금 전액
  - 일시 또는 계속지원
  - 대상:
    - 대한민국 국적 소지자 중 대학의 예술 및 체육계열 재학 중인 1학년 신입생
    - 전공탐색유형 또는 3학년 재학생
- **국가우수장학금(이공계)**
  - 등록금 전액
  - 일시 또는 계속지원
  - 대상:
    - 자연과학 및 공학계열 입학예정/확정 또는 재학 중인 자(휴학생 제외)
- **주거안정장학금**
  - 월 최대 20만원
  - 1학기
  - 대상:
    - 대한민국 국적 소지자 중 만 39세 이하 미혼이며 기초생활수급자 또는 차상위계층으로 확인된 자

또한 주거안정장학금의 경우:
- 신입생·편입생·재입학생은 **입학 첫 학기에 한하여 성적 및 이수학점 기준 미적용**
- 직전학기 성적 및 이수학점 기준이 별도로 제시되어 있습니다.
  - 장애인: 성적 및 이수학점 기준 제한 없음
  - 기초·차상위: C학점(70점/100점) 이상
  - 직전학기 이수학점: 12학점 이상

## 3) 계속지원기준이 있는 장학금
자료에서 계속지원 기준이 확인되는 장학금도 있습니다.

- **국가우수장학금(이공계)**  
  계속지원기준:
  - 직전학기 평점 3.5 이상 또는 백분위 87점 이상
  - 최소이수학점 직전학기 12학점 이상  
  (소속 대학 최저 이수학점 또는 최대 이수가능 학점이 12학점 미만인 경우 학사규정에 따름)

- **인문100년장학금 / 예술체육비전장학금**
  - 전공탐색유형: 성적 및 이수학점 기본자격 요건 없음
  - 전공확립유형: 직전 2년간 총 평균성적 백분위 90점 이상 또는 평점평균 3.6 이상, 취득 이수학점이 졸업이수학점의 40% 이상

## 4) 선발 방식
자료에 따르면 일부 장학금은:
- **대학별 배정인원 범위 내에서**
- **대학 자체 선발기준에 따라 선발**됩니다.

특히 인문100년장학금, 예술체육비전장학금, 국가우수장학금(이공계)은  
학업성적, 학생역량, 경제적 수준 등을 고려하여 선발하며, 장학금별로 세부 비율과 기준이 다릅니다.

---

## 참고한 문서
- **2026-2학기_학사안내_자료.pdf**
  - **페이지 40**: 교내장학금 규정
  - **페이지 44**: 국가장학금/주거안정장학금 규정

원하시면 제가 다음 단계로  
**“장학금별로 표 형태로 정리”**하거나,  
**“내가 받을 수 있는 장학금이 무엇인지 조건별로 확인”**해드릴 수 있습니다.


질문: 수강신청 기간에 대해 알려줘



수강신청 기간은 아래와 같습니다.  
(참고: **출처 2026-2학기_학사안내_자료.pdf, p.23**)

- **1차 장바구니 수강신청**: 2026. 7. 16.(목) ~ 7. 17.(금)  
  - 7. 16.(목) 10:00 ~ 23:30  
  - 7. 17.(금) 10:00 ~ 17:00

- **1차 수강신청**: 2026. 7. 22.(수) ~ 7. 24.(금)  
  - 7. 22.(수) 14:00 ~ 23:30  
  - 7. 23.(목) 10:00 ~ 23:30  
  - 7. 24.(금) 10:00 ~ 17:00

- **교차 수강신청**: 2026. 8. 4.(화) ~ 8. 5.(수)  
  - 8. 4.(화) 10:00 ~ 23:30  
  - 8. 5.(수) 10:00 ~ 17:00

- **2차 장바구니 수강신청**: 2026. 8. 18.(화)  
  - 10:00 ~ 17:00

- **2차 수강신청**: 2026. 8. 20.(목)  
  - 10:00 ~ 17:00

- **수강신청 정정 및 취소**: 2026. 9. 1.(화) ~ 9. 7.(월)  
  - 각 일자별 10:00 ~ 23:30  
  - 마지막 날은 10:00 ~ 17:00

- **수강신청 포기 기간**: 2026. 9. 17.(목) ~ 9. 18.(금)  
  - 각 일자별 10:00 ~ 23:30  
  - 마지막 날은 10:00 ~ 17:00

추가로, 자료에는 **상기 일정은 상황에 따라 변경될 수 있음**이라고 안내되어 있습니다.

**출처:** 2026-2학기_학사안내_자료.pdf, **p.23**

## 프로젝트 점검 체크리스트

**완료한 항목을 확인하세요:**

- [ ] PDF 문서 선정 및 로딩 완료
- [ ] Child Chunk 생성 완료
- [ ] Qdrant Cloud에 데이터 저장 완료
- [ ] Parent Document Retriever 구현 완료
- [ ] 검색 테스트 완료 (Child vs Parent 비교)
- [ ] RAG 시스템 구현 완료
- [ ] 최소 3개 이상의 질문으로 테스트 완료
- [ ] 시스템 프롬프트 도메인에 맞게 수정 완료

---

## 추가 개선 아이디어

1. **청킹 전략 최적화**: chunk_size와 chunk_overlap 조정
2. **검색 개수 조정**: retriever의 k 값 변경
3. **프롬프트 개선**: 더 구체적인 답변 형식 지정
4. **메타데이터 활용**: 날짜, 카테고리 등 추가 필터링
5. **하이브리드 검색**: 키워드 + 벡터 검색 결합